In [ ]:
from setup import *

## get all entities

Entities sind die angebundenen RIS. Mit diesem Code-Block ziehen wir uns alle entities mit Metadaten und ids.

Diese Liste ist nötig, um später gezielt für bestimmte Kommunen Daten zu finden, oder Rechercheergebnisse passend zu filtern.


In [ ]:
# Alle Items sammeln
all_entities = []

# Paginierungsparameter
#achtung, tatsächlich ist gerade auf 300 Limit
limit = 500
offset = 0
total = 16079

print(f"{poliscope_api_url}/entities")

# Durch alle Pages iterieren
while offset < total:
    response = requests.get(
        url=f"{poliscope_api_url}/entities",
        headers=poliscope_headers,
        params={
            "limit": limit,
            "offset": offset,
            "detail": "standard",
        }
    )
    
    if response.status_code == 200:
        data = response.json()
        items = data.get("data", [])
        all_entities.extend(items)  # Alle Items zur Liste hinzufügen
        
        print(f"Downloaded {offset + len(items)} / {total} items")
        offset += limit
    else:
        print(f"Error: {response.status_code}")
        break

In [ ]:
entities_df = pd.DataFrame(all_entities)
entities_df

In [ ]:
entities_df.to_csv("./data/metadata/all_entities.csv", index=False)

# Großstädte filtern

Basierend auf dem entities-dataframe können wir bestimmte Gruppen von Städten filtern, etwa Großstädte. Dazu können auch andere Datenquellen herangezogen werden, je nachdem welcher Filter gewünscht ist.

10 - Bundesländer

40 - Landkreise

50 - Gemeindeverbände

60 - Gemeinden

PR - Planungsregionen

id entspricht dem ARS Schlüssel

In [ ]:
big_cities = entities_df[(entities_df["population"] >= 100000)].copy()
big_cities = big_cities[big_cities["ris"].notnull()]

In [ ]:
big_cities = big_cities[big_cities["level"].isin(["60", "50", "10"])]
big_cities = big_cities[~big_cities["parents"].apply(lambda x: x[0]["name"] if x else None).isin(["Berlin, Stadt", "Hamburg, Freie und Hansestadt"])]

In [ ]:
big_cities.to_csv("./data/raw/big_cities.csv", index=False)